In [9]:
import re
import os
import time
import requests
import oci
import pandas as pd
from oci.ai_language import AIServiceLanguageClient
from oci.ai_language.models import TextDocument, BatchDetectLanguageSentimentsDetails

print("✅ Librerías importadas")

✅ Librerías importadas


In [2]:
CONFIG_PATH = "/Workspace/oci/config"
BEARER_TOKEN = "AAAAAAAAAAAAAAAAAAAAANuH8AEAAAAA6wKuTd3d7RF2NJn0YHtxKN9zwLA%3DluStWS7qPQxjJLmEqk2TZs3pew2TpZjYgX868PydNMa4tbdXNp"
LANGUAGE_CODE = "en"
MAX_REPLIES = 10
TWEET_URL = "https://x.com/Tesla/status/2029247781559521549"

config = oci.config.from_file(CONFIG_PATH, "DEFAULT")
print(f"✅ Config OCI cargado - Region: {config['region']}")

✅ Config OCI cargado - Region: us-chicago-1


In [13]:
def analyze_sentiment_oci(client, text, key="doc1"):
    try:
        doc = TextDocument(key=key, text=text, language_code=LANGUAGE_CODE)
        response = client.batch_detect_language_sentiments(
            batch_detect_language_sentiments_details=BatchDetectLanguageSentimentsDetails(documents=[doc]),
            level=["SENTENCE"]
        )
        if response.data and response.data.documents:
            result = response.data.documents[0]
            return {
                'sentiment': result.document_sentiment,
                'scores': {
                    'positive': result.document_scores.get('Positive', 0),
                    'negative': result.document_scores.get('Negative', 0),
                    'neutral': result.document_scores.get('Neutral', 0),
                    'mixed': result.document_scores.get('Mixed', 0)
                }
            }
        return None
    except Exception as e:
        print(f"Error OCI: {e}")
        return None

print("✅ Funciones OCI definidas")

✅ Funciones OCI definidas


In [6]:
def get_headers():
    return {"Authorization": f"Bearer {BEARER_TOKEN}", "Content-Type": "application/json"}

def extract_tweet_id(url):
    for pattern in [r'status/(\d+)', r'twitter\.com/[^/]+/status/(\d+)', r'x\.com/[^/]+/status/(\d+)']:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

def get_tweet(tweet_id):
    url = f"https://api.twitter.com/2/tweets/{tweet_id}"
    params = {"tweet.fields": "created_at,public_metrics,conversation_id,text"}
    try:
        response = requests.get(url, headers=get_headers(), params=params)
        return response.json().get("data") if response.status_code == 200 else None
    except:
        return None

def get_replies(conversation_id, max_results=10):
    url = "https://api.twitter.com/2/tweets/search/recent"
    params = {
        "query": f"conversation_id:{conversation_id} -is:retweet",
        "max_results": min(max_results, 100),
        "tweet.fields": "created_at,text,author_id",
        "expansions": "author_id",
        "user.fields": "username,name"
    }
    try:
        response = requests.get(url, headers=get_headers(), params=params)
        if response.status_code == 200:
            data = response.json()
            replies = [r for r in data.get("data", []) if r.get("id") != conversation_id]
            
            # Mapear usuarios
            users = {}
            if "includes" in data and "users" in data["includes"]:
                for user in data["includes"]["users"]:
                    users[user["id"]] = {
                        "username": user.get("username", "unknown"),
                        "name": user.get("name", "Unknown User")
                    }
            
            # Agregar info de usuario a cada reply
            for reply in replies:
                author_id = reply.get("author_id")
                if author_id in users:
                    reply["username"] = users[author_id]["username"]
                    reply["user_name"] = users[author_id]["name"]
                else:
                    reply["username"] = "unknown"
                    reply["user_name"] = "Unknown User"
            
            return replies
        return []
    except:
        return []

def anonymize_text(text):
    text = re.sub(r'@\w+', '[USUARIO]', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'http[s]?://\S+', '[URL]', text)
    return text

print("✅ Funciones X API definidas")

✅ Funciones X API definidas


In [ ]:
def analyze_tweet_replies(tweet_url, max_replies=10):
    """Analiza replies de un tweet y devuelve DataFrame ordenado por toxicidad"""
    
    # Crear cliente OCI
    oci_client = AIServiceLanguageClient(config)
    
    # Extraer tweet_id
    tweet_id = extract_tweet_id(tweet_url)
    if not tweet_id:
        return None
    
    # Obtener tweet
    tweet = get_tweet(tweet_id)
    if not tweet:
        return None
    
    # Obtener replies
    conversation_id = tweet.get('conversation_id', tweet_id)
    replies = get_replies(conversation_id, max_replies)
    
    if not replies:
        return None
    
    # Analizar cada reply
    analyzed = []
    for reply in replies:
        text = reply.get('text', '')
        username = reply.get('username', 'unknown')
        user_name = reply.get('user_name', 'Unknown User')
        created_at = reply.get('created_at', 'N/A')
        reply_id = reply.get('id', '')
        
        oci_result = analyze_sentiment_oci(oci_client, text, reply_id)
        
        if oci_result:
            scores = oci_result['scores']
            neg = scores['negative']
            pos = scores['positive']
            neu = scores['neutral']
            sentiment = oci_result['sentiment']
            
            # Clasificar basado en score de OCI
            if neg > 0.7:
                cat, lvl = "Hate Speech", "high"
            elif neg > 0.5:
                cat, lvl = "Offensive Language", "medium"
            elif neg > 0.3:
                cat, lvl = "Harassment", "low"
            else:
                cat, lvl = "Acceptable", "low"
            
            analyzed.append({
                'reply_id': reply_id,
                'username': username,
                'user_name': user_name,
                'text': text,
                'category': cat,
                'severity_level': lvl,
                'toxicity_score': round(neg * 10, 2),
                'oci_sentiment': sentiment,
                'oci_negative': neg,
                'oci_positive': pos,
                'oci_neutral': neu,
                'created_at': created_at                
            })
            
        time.sleep(2.0)
    
    # Crear DataFrame y ordenar por toxicidad (descendente)
    df = pd.DataFrame(analyzed)
    df = df.sort_values('toxicity_score', ascending=False).reset_index(drop=True)
    
    return df

print("✅ Función principal definida")

✅ Función principal definida


In [17]:
df_resultados= analyze_tweet_replies(TWEET_URL, max_replies=MAX_REPLIES)
df_resultados

,reply_id,username,user_name,text,category,severity_level,toxicity_score,oci_sentiment,oci_negative,oci_positive,oci_neutral,created_at
0,2031061318476697626,Ilannasltd,Ilannas LTD,@Tesla @Tesla_Optimus They need one of those l...,Acceptable,low,0.77,Neutral,0.077179,0.028815,0.852458,2026-03-09T17:35:28.000Z
1,2030906381952553286,Ahiala96,ToMarcoxxxx,"@Tesla @Tesla_Optimus Release it, don’t make p...",Acceptable,low,0.77,Neutral,0.077461,0.022833,0.762493,2026-03-09T07:19:48.000Z
2,2031062177763963056,quaranislight,Qur'an,@Tesla @Tesla_Optimus https://t.co/MVao8XchAS,Acceptable,low,0.70,Neutral,0.070222,0.035869,0.758164,2026-03-09T17:38:53.000Z
3,2031064387390238915,travelwh,Finance Fortune Cookie 🇺🇸 ⛪️ ⛷ 🏒 🐕‍🦺,"@Tesla @Tesla_Optimus forget the turing , ult...",Acceptable,low,0.34,Neutral,0.033832,0.021216,0.842333,2026-03-09T17:47:39.000Z
4,2030883721193402550,ng_jason,Jason Ng,"@Tesla If there is a day where we need to ""pul...",Acceptable,low,0.33,Neutral,0.033376,0.014707,0.951917,2026-03-09T05:49:45.000Z
5,2031095182263742627,NWilliams52017,Janet Alex,@Tesla Hey,Acceptable,low,0.25,Neutral,0.025487,0.058998,0.881520,2026-03-09T19:50:02.000Z
6,2031179024949707061,BuzzDalways,Buzz Dalways®,"@Tesla Will, Optimus have the ability to walk ...",Acceptable,low,0.22,Neutral,0.022456,0.176612,0.737816,2026-03-10T01:23:11.000Z
7,2031087248746123503,Space_master24,Space Master,@Tesla Thank you for your incredible love and ...,Acceptable,low,0.21,Positive,0.020516,0.706604,0.194623,2026-03-09T19:18:30.000Z
8,2031163351615865119,MildredBunwh7j,MildredBunn,@Tesla I want to buy Tesla,Acceptable,low,0.18,Positive,0.018150,0.627400,0.314313,2026-03-10T00:20:54.000Z
9,2031081709840605503,ElonMusk304964,Elon Musk,"@rjmad465088 @Tesla Congratulations 🎊 🎉 fan, y...",Acceptable,low,0.08,Neutral,0.008272,0.078857,0.879802,2026-03-09T18:56:29.000Z


In [176]:
# Convertir pandas DataFrame a Spark DataFrame
spark_df = spark.createDataFrame(df_resultados)
spark_df

DataFrame[reply_id: string, username: string, user_name: string, text: string, category: string, severity_level: string, toxicity_score: double, oci_sentiment: string, oci_negative: double, oci_positive: double, oci_neutral: double, created_at: string]

In [178]:
# Crear catalog y schema para el análisis de hate speech
spark.sql("CREATE CATALOG IF NOT EXISTS analysis")
spark.sql("CREATE SCHEMA IF NOT EXISTS analysis.tweets")

print("✅ Catalog y schema creados: analysis.tweets")

✅ Catalog y schema creados: analysis.tweets


In [186]:
# Crear tabla Delta con clustering (optimización automática)
spark.sql("""
CREATE TABLE IF NOT EXISTS analysis.tweets.interactions_analysis (
    reply_id STRING,
    username STRING,
    user_name STRING,
    text STRING,
    category STRING,
    severity_level STRING,
    toxicity_score DOUBLE,
    oci_sentiment STRING,
    oci_negative DOUBLE,
    oci_positive DOUBLE,
    oci_neutral DOUBLE,
    created_at STRING    
)
USING DELTA
CLUSTER BY (category, toxicity_score)
COMMENT 'Análisis de discurso de odio en replies de X/Twitter usando OCI AI Language'
""")

print("✅ Tabla Delta creada: analysis.tweets.interactions_analysis")

# Insertar datos en la tabla
spark_df.write.option('write.mode', 'MERGE').option('write.merge.keys','reply_id').insertInto('analysis.tweets.interactions_analysis')

print(f"✅ {len(df_resultados)} registros insertados en la tabla")

✅ Tabla Delta creada: analysis.tweets.interactions_analysis


✅ 10 registros insertados en la tabla


In [187]:
# Leer desde la tabla
df_delta = spark.sql("SELECT * FROM analysis.tweets.interactions_analysis")

# Mostrar estadísticas
print("\n📊 ESTADÍSTICAS DE LA TABLA")
print(f"Total de registros: {df_delta.count()}")

# Distribución por categoría
print("\nDistribución por categoría:")
spark.sql("""
    SELECT category, COUNT(*) as total, AVG(toxicity_score) as avg_toxicity
    FROM analysis.tweets.interactions_analysis
    GROUP BY category
    ORDER BY avg_toxicity DESC
""").show()

# Top 10 comentarios más tóxicos
print("\nTop 10 comentarios más tóxicos:")
spark.sql("""
    SELECT username, user_name, category, toxicity_score, 
           SUBSTRING(text, 1, 50) as text_preview
    FROM analysis.tweets.interactions_analysis
    WHERE category != 'Acceptable'
    ORDER BY toxicity_score DESC
    LIMIT 10
""").show(truncate=False)

# Convertir a pandas si es necesario
df_pandas = df_delta.toPandas()
print(f"\n✅ Datos disponibles en Spark y Pandas")


📊 ESTADÍSTICAS DE LA TABLA


Total de registros: 10

Distribución por categoría:


+------------------+-----+------------------+
|          category|total|      avg_toxicity|
+------------------+-----+------------------+
|Offensive Language|    2|              6.41|
|        Harassment|    1|              3.13|
|        Acceptable|    7|1.6671428571428568|
+------------------+-----+------------------+


Top 10 comentarios más tóxicos:


+--------------+-------------+------------------+--------------+--------------------------------------------------+
|username      |user_name    |category          |toxicity_score|text_preview                                      |
+--------------+-------------+------------------+--------------+--------------------------------------------------+
|Yoda_Mandering|Jerry Mantell|Offensive Language|6.48          |@IanMalcolm84 @elonmusk Are you trying to get a co|
|IBUrlegna4U   |Yesi         |Offensive Language|6.34          |@elonmusk Whatever we don’t do now we will regret |
|FlyStar231    |Aliwise™     |Harassment        |3.13          |@elonmusk So political parties are now pay back st|
+--------------+-------------+------------------+--------------+--------------------------------------------------+




✅ Datos disponibles en Spark y Pandas


In [189]:
# Using Spark SQL
df = spark.sql("SELECT * FROM ora26ai.app_aidp.interactions_analysis")
df

DataFrame[reply_id: string, username: string, user_name: string, text: string, category: string, severity_level: string, toxicity_score: decimal(10,2), oci_sentiment: string, oci_negative: decimal(10,4), oci_positive: decimal(10,4), oci_neutral: decimal(10,4), created_at: string]

In [190]:
spark_df.write.option('write.mode', 'MERGE').option('write.merge.keys','reply_id').insertInto('ora26ai.app_aidp.interactions_analysis')
spark_df.show()

+-------------------+--------------+-----------------+--------------------+------------------+--------------+--------------+-------------+------------+------------+-----------+--------------------+
|           reply_id|      username|        user_name|                text|          category|severity_level|toxicity_score|oci_sentiment|oci_negative|oci_positive|oci_neutral|          created_at|
+-------------------+--------------+-----------------+--------------------+------------------+--------------+--------------+-------------+------------+------------+-----------+--------------------+
|2012792305623728565|Yoda_Mandering|    Jerry Mantell|@IanMalcolm84 @el...|Offensive Language|        medium|          6.48|     Negative|  0.64804655| 0.041376896| 0.23015478|2026-01-18T07:40:...|
|2012790989841182947|   IBUrlegna4U|             Yesi|@elonmusk Whateve...|Offensive Language|        medium|          6.34|     Negative|   0.6342297| 0.015538869|  0.3072976|2026-01-18T07:35:...|
|201278831